# Удержание: на чём держится выручка 2011 года

Вопрос, с которого начался разбор: новых клиентов приходит вдвое меньше, чем годом ранее, а
выручка не падает. Проверяю четыре вещи — когорты, интервалы между покупками, сегменты RFM и
концентрацию выручки.

Данные — `data/curated/fact_transactions.parquet`, витрины — `data/curated/marts/`
(`python -m retail_analytics.marts`). Ноутбук запускается из корня репозитория.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from retail_analytics import config as cfg, metrics as m, prepare
from retail_analytics.marts import MARTS_DIR

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 44)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
kpi = pd.read_parquet(MARTS_DIR / "monthly_kpi.parquet")
kpi["year"] = kpi.month.astype(str).str[:4]
kpi.groupby("year")[["revenue", "orders", "new_customers", "returning_customers"]].sum().round(0)

,revenue,orders,new_customers,returning_customers
year,,,,
2009,"798,119.00",1821,951,0
2010,"9,375,309.00",20629,3364,9049
2011,"9,470,438.00",18877,1538,10602


## Год к году

2010 и 2011 по обороту почти одинаковые, но собраны разными способами: заказов на 8,5% меньше,
впервые купивших — в 2,2 раза меньше, а вернувшихся — на 17% больше. Средний чек при этом +10%:
оборот держат не новые покупатели, а более крупные покупки старых. 2009 год в таблице — только
декабрь, сравнивать его нельзя.

In [3]:
fig = px.bar(kpi, x="month", y=["new_customers", "returning_customers"])
fig.update_layout(height=340, margin=dict(t=30))
fig.show()

In [4]:
cols = ["revenue", "orders", "new_customers", "returning_customers"]
yoy = kpi[kpi.year.isin(["2010", "2011"])].groupby("year")[cols].sum().T
yoy.loc["aov"] = yoy.loc["revenue"] / yoy.loc["orders"]
yoy["изменение, %"] = ((yoy["2011"] / yoy["2010"] - 1) * 100).round(1)
yoy[["2010", "2011", "изменение, %"]]

year,2010,2011,"изменение, %"
revenue,"9,375,309.08","9,470,438.33",1.00
orders,"20,629.00","18,877.00",-8.50
new_customers,"3,364.00","1,538.00",-54.30
returning_customers,"9,049.00","10,602.00",17.20
aov,454.47,501.69,10.40


## Когорты

Когорта — месяц первой покупки идентифицированного клиента. Активный месяц засчитывается один
раз, сколько бы покупок в нём ни было.

In [5]:
coh = pd.read_parquet(MARTS_DIR / "cohort_retention.parquet")
matrix = m.cohort_pivot(coh)
fig = px.imshow(matrix, text_auto=".0%", aspect="auto", color_continuous_scale="Blues")
fig.update_layout(height=520)
fig.show()

In [6]:
coh.groupby("month_number").retention.mean().round(3).head(13)

month_number
0    1.00
1    0.21
2    0.22
3    0.22
4    0.21
5    0.19
6    0.18
7    0.18
8    0.16
9    0.15
10   0.15
11   0.17
12   0.18
Name: retention, dtype: float64

Среднее удержание падает до 21% уже на втором месяце и к восьмому-девятому доходит до 15%.
При этом колонки когорт читаются неровно: у зимних когорт сезонный старт выше, у летних — ниже, и
это надо помнить, когда когорты усредняют.

Теперь те же когорты в деньгах. Считается доля выручки когорты по месяцам её жизни.

In [7]:
crev = pd.read_parquet(MARTS_DIR / "cohort_revenue.parquet")
curve = crev.groupby("month_number").revenue_share.mean().round(4).head(24)
fig = px.line(x=curve.index, y=curve.values, markers=True,
              labels={"x": "Месяц после первой покупки", "y": "Доля выручки когорты"})
fig.update_traces(line_color="#2f6f8f", line_width=3)
fig.update_layout(height=340)
fig.show()
curve

month_number
0    0.38
1    0.07
2    0.07
3    0.07
4    0.06
5    0.05
6    0.04
7    0.08
8    0.04
9    0.04
10   0.04
11   0.05
12   0.06
13   0.04
14   0.04
15   0.04
16   0.04
17   0.04
18   0.03
19   0.04
20   0.04
21   0.04
22   0.04
23   0.03
Name: revenue_share, dtype: float64

Разрыв между «удержали» и «приносят деньги» виден здесь: в клиентах когорта теряет четыре
пятых за месяц, а в деньгах после стартового месяца (37,5% выручки когорты) держится ровные
3–8% на каждом следующем — вплоть до двадцатого. Уходит мелочь: те, кто остался, покупают
стабильно.

## Интервал между покупками

По нему выбирают окно реактивации, поэтому квантили, а не среднее: у клиентов, купивших один раз
за два года, интервал просто огромный.

In [8]:
gaps = pd.read_parquet(MARTS_DIR / "purchase_gaps.parquet")
gaps.T

,0
gaps_observed,"30,744.00"
avg_gap_days,52.10
p25_gap_days,7.00
median_gap_days,25.00
p75_gap_days,62.00
share_gaps_up_to_30d,0.55
share_gaps_over_180d,0.06


In [9]:
view = m.apply_filters(m.load_fact(cfg.FACT_FILE))
dist = m.purchase_gaps(view).clip(upper=365)
fig = px.histogram(dist, nbins=48, labels={"value": "Дней между покупками", "count": "Пар покупок"})
fig.add_vline(x=float(dist.median()), line_dash="dot", line_color="#b8574f",
              annotation_text=f"медиана {dist.median():.0f}")
fig.update_layout(height=320)
fig.show()

Медиана — 25 дней, P75 — 62. Больше половины интервалов укладываются в месяц, а дольше
полугода — 6,4%. То есть «рабочий цикл» покупателя здесь около месяца, всё, что дальше квартала,
— это уже не длинный цикл, а человек, которого надо возвращать.

## RFM и сегменты

Баллы считаются по квантильным порогам, а не по `NTILE`. Частота покупки принимает мало
значений, и на границе квантиля стоит сразу несколько сотен одинаковых клиентов: `NTILE`
раздавал им разные баллы по порядку строк, и на пересборке витрины 33 человека уезжали в
другой сегмент. С порогами группы получаются неровными (1 619 / 945 / 1 148 / 1 024 /
1 117 по частоте), зато один и тот же клиент не может попасть в два сегмента.

In [10]:
seg = pd.read_parquet(MARTS_DIR / "rfm_segments.parquet")
seg.round(4)

,segment,customers,customers_share,revenue,revenue_share,avg_revenue_per_customer,avg_orders,avg_recency_days
0,ядро,1258,0.21,"11,580,350.55",0.68,"9,205.37",17.31,19.00
1,остывают,742,0.13,"1,963,933.53",0.12,"2,646.81",6.86,106.00
2,дорогой отток,394,0.07,"1,408,913.32",0.08,"3,575.92",6.39,365.00
3,отток,1945,0.33,"806,727.80",0.05,414.77,1.80,437.00
4,лояльные,580,0.10,"634,810.19",0.04,"1,094.50",4.03,24.00
5,свежие,513,0.09,"443,137.61",0.03,863.82,1.53,27.00
6,прочее,421,0.07,"228,957.87",0.01,543.84,1.42,107.00


In [11]:
fig = px.bar(
    pd.DataFrame({
        "сегмент": list(seg.segment) * 2,
        "доля": list(seg.customers_share) + list(seg.revenue_share),
        "метрика": ["клиентов"] * len(seg) + ["выручки"] * len(seg),
    }),
    x="доля", y="сегмент", color="метрика", barmode="group",
    color_discrete_map={"клиентов": "#a3829c", "выручки": "#2f6f8f"},
)
fig.update_layout(height=400, xaxis=dict(tickformat=".0%"), yaxis=dict(autorange="reversed"))
fig.show()

Ядро — 21,5% клиентов и 67,9% выручки. Отток — треть клиентов и 4,7% выручки:
возвращать его было бы нечем. Между ними две группы, на которых стоит вся работа по
коммуникациям, — «остывают» (742 клиента, £1,96 млн, в среднем 106 дней тишины) и «дорогой
отток» (394 клиента, £1,41 млн, 365 дней тишины).

In [12]:
rfm = pd.read_parquet(MARTS_DIR / "rfm.parquet")
fig = px.scatter(rfm.sample(4000, random_state=7), x="recency_days", y="monetary",
                 color="segment", log_y=True, opacity=0.55)
fig.update_layout(height=420)
fig.show()

## Концентрация выручки

Сколько клиентов держат половину и восемьдесят процентов оборота.

In [13]:
conc = pd.read_parquet(MARTS_DIR / "customer_concentration.parquet")
conc.T


,0
customers_for_half_of_revenue,258.00
customers_for_80_of_revenue,"1,353.00"
total_customers,"5,853.00"
share_for_50pct,0.04
share_for_80pct,0.23


In [14]:
pareto = m.pareto(view)
pareto[["rank", "customer_id", "revenue", "customer_share", "cum_share"]].head(8)

,rank,customer_id,revenue,customer_share,cum_share
0,1,18102,"580,987.04",0.00,0.03
1,2,14646,"526,751.52",0.00,0.06
2,3,14156,"303,069.88",0.00,0.08
3,4,14911,"272,252.79",0.00,0.10
4,5,17450,"244,784.25",0.00,0.11
5,6,13694,"195,640.69",0.00,0.12
6,7,17511,"172,132.87",0.00,0.13
7,8,16446,"168,472.50",0.00,0.14


258 клиентов из 5 853 (4,4%) дают половину выручки, 1 353 (23%) — 80%. Это не «топ-20
клиентов», как обычно рисуют в слайдах: чтобы набрать 50%, нужен каждый двадцатый покупатель
базы.

## Что из этого следует

- Расти дальше за счёт новых клиентов нельзя: приток 2011 года — 128 человек в месяц против 280
  в 2010-м. Вся динамика держится на повторных покупках.
- Вторичная покупка решается в первые два месяца: удержание сползает с 21% до 15% к
  восьмому-девятому месяцу, дальше кривая почти плоская.
- «Дорогой отток» — единственная группа, где возврат окупается без скидки: £3 576 среднего
  вклада на человека. Но писать надо по имени и по товарной истории, а не рассылкой по базе.
- Чего эти данные не отвечают: себестоимости, рекламы и доставки по складу нет, поэтому маржу и
  CAC посчитать нельзя. Выводы ограничены оборотом и поведением покупателей.